### Getting the chromothripsis positive samples from ShatterSeek summary txt file


#### Individual sample high confidence detection

In [ ]:
import pandas as pd
import os

def identify_chromothripsis(df):
    """
    Identify high-confidence chromothripsis events per chromosome.
    Returns a list of dicts with chromosome, criteria met, and supporting values.
    """
    results = []

    for _, row in df.iterrows():
        info = dict(row)
        chrom = info['chrom']

        intrachromosomal_svs = (info['number_DEL'] + info['number_DUP'] +
                                info['number_h2hINV'] + info['number_t2tINV'])
        interchromosomal_svs = info['number_TRA']
        cn_oscillations = info['max_number_oscillating_CN_segments_2_states']
        fragment_joins_significant = info['pval_fragment_joins'] < 0.05
        chr_enrichment_significant = (info['chr_breakpoint_enrichment'] < 0.05
                                      if info['chr_breakpoint_enrichment'] != 'NA' else False)
        exp_distribution_significant = (info['pval_exp_chr'] < 0.05
                                        if info['pval_exp_chr'] != 'NA' else False)

        supporting_values = {
            'intrachromosomal_svs': intrachromosomal_svs,
            'interchromosomal_svs': interchromosomal_svs,
            'cn_oscillating_segments_2states': cn_oscillations,
            'pval_fragment_joins': info['pval_fragment_joins'],
            'chr_breakpoint_enrichment': info['chr_breakpoint_enrichment'],
            'pval_exp_chr': info['pval_exp_chr'],
            'region': f"chr{chrom}:{info['start']}-{info['end']}",
        }

        # High confidence criterion 1:
        # >=6 interleaved intrachromosomal SVs, >=7 CN oscillating segments,
        # fragment joins test, and (chr enrichment OR exp distribution)
        high_conf_1 = (intrachromosomal_svs >= 6 and
                       cn_oscillations >= 7 and
                       fragment_joins_significant and
                       (chr_enrichment_significant or exp_distribution_significant))

        # High confidence criterion 2:
        # >=3 intrachromosomal SVs AND >=4 interchromosomal SVs,
        # >=7 CN oscillating segments, fragment joins test
        high_conf_2 = (intrachromosomal_svs >= 3 and
                       interchromosomal_svs >= 4 and
                       cn_oscillations >= 7 and
                       fragment_joins_significant)

        criteria_met = []
        if high_conf_1:
            criteria_met.append('High_confidence_1 (>=6 intra SVs, >=7 CN osc, fragment joins, chr enrichment/exp dist)')
        if high_conf_2:
            criteria_met.append('High_confidence_2 (>=3 intra + >=4 inter SVs, >=7 CN osc, fragment joins)')

        if criteria_met:
            results.append({
                'chrom': chrom,
                'criteria_met': criteria_met,
                'values': supporting_values,
            })

    return results


def process_folder(folder_path):
    all_results = {}

    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith("_summary.txt"):
            file_path = os.path.join(folder_path, filename)
            sample_name = filename.replace("_summary.txt", "")

            df = pd.read_csv(file_path, sep='\t')
            ct_results = identify_chromothripsis(df)

            if ct_results:
                all_results[sample_name] = ct_results

    return all_results


def print_results(all_results):
    if not all_results:
        print("No samples with high-confidence chromothripsis detected.")
        return

    print("=" * 90)
    print("CHROMOTHRIPSIS DETECTION RESULTS (High Confidence Only)")
    print("=" * 90)

    for sample, events in all_results.items():
        print(f"\n{'─' * 90}")
        print(f"Sample: {sample}")
        print(f"  Chromothripsis events found: {len(events)}")
        chroms = [str(e['chrom']) for e in events]
        print(f"  Chromosomes: {', '.join(chroms)}")

        for event in events:
            v = event['values']
            print(f"\n  Chromosome {event['chrom']}")
            print(f"    Region: {v['region']}")
            print(f"    Intrachromosomal SVs:          {v['intrachromosomal_svs']}")
            print(f"    Interchromosomal SVs (TRA):    {v['interchromosomal_svs']}")
            print(f"    CN oscillating segments (2st):  {v['cn_oscillating_segments_2states']}")
            print(f"    p-val fragment joins:           {v['pval_fragment_joins']}")
            print(f"    Chr breakpoint enrichment:      {v['chr_breakpoint_enrichment']}")
            print(f"    p-val exp distribution (chr):   {v['pval_exp_chr']}")
            print(f"    Criteria matched:")
            for c in event['criteria_met']:
                print(f"      - {c}")

    print(f"\n{'=' * 90}")
    print(f"Summary: {len(all_results)} sample(s) with chromothripsis detected")
    total_events = sum(len(e) for e in all_results.values())
    print(f"         {total_events} total chromothripsis event(s) across all samples")
    print("=" * 90)


def save_results_tsv(all_results, output_file):
    """Save results as a TSV for downstream analysis."""
    rows = []
    for sample, events in all_results.items():
        for event in events:
            v = event['values']
            rows.append({
                'sample': sample,
                'chrom': event['chrom'],
                'region': v['region'],
                'intrachromosomal_svs': v['intrachromosomal_svs'],
                'interchromosomal_svs': v['interchromosomal_svs'],
                'cn_oscillating_2states': v['cn_oscillating_segments_2states'],
                'pval_fragment_joins': v['pval_fragment_joins'],
                'chr_breakpoint_enrichment': v['chr_breakpoint_enrichment'],
                'pval_exp_chr': v['pval_exp_chr'],
                'criteria_met': '; '.join(event['criteria_met']),
            })

    if rows:
        out_df = pd.DataFrame(rows)
        out_df.to_csv(output_file, sep='\t', index=False)
        print(f"\nResults saved to: {output_file}")
    else:
        print("No results to save.")


# ── Main ──────────────────────────────────────────────────────────────────────
folder_path = '/path/to/shatterseek/results'   # folder with ShatterSeek <sample>_summary.txt files
output_file = os.path.join(folder_path, 'chromothripsis_summary.tsv')

all_results = process_folder(folder_path)
print_results(all_results)
save_results_tsv(all_results, output_file)